### ***INT4(W4A16) GPTQ (LLM Compressor) 스크립트***

In [ ]:
# =========================================================
# LG Aimers Hackathon - EXAONE-4.0-1.2B 제출용
# INT4(W4A16) GPTQ (LLM Compressor) "정확도(점수) 우선" 최종 스크립트 v4.1
#
# v4.1 핵심 수정(점수 급락 방지):
#  1) ✅ tokenizer.truncation_side="left" 를 "저장물(tokenizer_config.json)에도 반영" (런타임 tail 보존)
#     - 길어진 프롬프트에서 질문이 잘리는 문제 방지 (점수 급락의 가장 흔한 원인)
#  2) ✅ concatenate_data=False 기본 (샘플 수/다양성 보존 → GPTQ 품질 개선)
#     - batch_size=1이면 concat의 이득이 작고, 오히려 샘플 수가 줄어들 수 있음
#  3) ✅ GPTQModifier.scheme에 dict 직접 넣지 않고, config_groups로 weights.group_size 명시
#  4) ✅ actorder="static" 유지
#  5) ✅ oneshot 인자 "조용한 드랍" 금지 + 저장 산출물 검증 강화
#
# 주의:
#  - transformers 4.57.3 + 특정 ByteLevel tokenizer 조합에서 fix_mistral_regex가 TypeError 낼 수 있어
#    fix_mistral_regex는 절대 켜지 않음.
# =========================================================

# ---------------------------
# 0) Install (Colab에서 1번만)
# ---------------------------
!pip -q install -U \
  "transformers==4.57.3" \
  "datasets==4.4.1" \
  "accelerate==1.10.1" \
  "safetensors==0.7.0" \
  "llmcompressor==0.9.0.1"

import os, shutil, json, time, zipfile, inspect, gc, random, re
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

import llmcompressor
from llmcompressor import oneshot

# GPTQModifier import (경로 차이 방어)
try:
    from llmcompressor.modifiers.quantization.gptq import GPTQModifier
except Exception:
    try:
        from llmcompressor.modifiers.quantization.gptq.base import GPTQModifier
    except Exception:
        from llmcompressor.modifiers.quantization import GPTQModifier

from safetensors.torch import safe_open


# ---------------------------
# 1) Paths (여기만 본인 경로로 수정)
# ---------------------------
MODEL_ID = "/content/drive/MyDrive/data/lg_8th/base_model"
OUT_DIR_LOCAL = "/content/model_quant"

ZIP_NAME = "submit"
DRIVE_SAVE_DIR = "/content/drive/MyDrive/data/lg_8th/try0216_submit_v41"
DRIVE_MODEL_DIR = os.path.join(DRIVE_SAVE_DIR, "model")
DRIVE_ZIP_PATH  = os.path.join(DRIVE_SAVE_DIR, f"{ZIP_NAME}.zip")


# ---------------------------
# 2) Dataset / Calibration / Quant 설정 (Score-up 핵심)
# ---------------------------
DATASET_ID    = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"
SEED          = 42

PROFILE = "H100_SCORE"   # "H100_SCORE" / "H100_GOOD"

PROFILE_CFG = {
    # ✅ 점수 우선(권장)
    "H100_SCORE": {
        # concat을 끄면 실제 forward pass가 늘어날 수 있으니,
        # 너무 큰 값이 부담이면 8192로 낮추고 테스트해보세요.
        "num_calib": 32768,
        "max_len": 4096,
        "group_size": 32,     # 호환성/정확도 밸런스 (vLLM 호환 이슈 피하려면 32 유지 권장)
        "block_size": 128,    # GPTQ chunk (group_size와 별개)
        "dampening_frac": 0.01,
        "calib_batch_size": 1,
        "actorder": "static",
        "symmetric": True,    # ✅ 우선 True로 유지. (아래 팁: 점수 낮으면 False 실험)
        "concatenate_data": False,   # ✅ v4.1: 샘플 수/다양성 보존
        "pad_to_max_length": False,
        # long이 부족하면 자동으로 다른 길이로 채우므로 현실적인 비율 권장
        "len_mix": (0.25, 0.55, 0.20),
        # PROMPT가 보통 점수 안정적 (평가 입력이 "질문+지시" 중심일 때)
        "text_mode": "MIX",   # "PROMPT"/"FULL"/"MIX"
        # 한국어가 평가에서 중요하면 True 추천 (부족하면 자동 fallback)
        "korean_focus": False,
        "korean_min_ratio": 0.05,
    },

    # ✅ 빠르게 돌려볼 때
    "H100_GOOD": {
        "num_calib": 8192,
        "max_len": 4096,
        "group_size": 32,
        "block_size": 128,
        "dampening_frac": 0.01,
        "calib_batch_size": 1,
        "actorder": "static",
        "symmetric": True,
        "concatenate_data": False,
        "pad_to_max_length": False,
        "len_mix": (0.25, 0.55, 0.20),
        "text_mode": "PROMPT",
        "korean_focus": True,
        "korean_min_ratio": 0.08,
    },
}

NUM_CALIBRATION_SAMPLES = int(PROFILE_CFG[PROFILE]["num_calib"])
MAX_SEQUENCE_LENGTH     = int(PROFILE_CFG[PROFILE]["max_len"])
CALIB_BATCH_SIZE        = int(PROFILE_CFG[PROFILE]["calib_batch_size"])
GPTQ_GROUP_SIZE         = int(PROFILE_CFG[PROFILE]["group_size"])
GPTQ_BLOCK_SIZE         = int(PROFILE_CFG[PROFILE]["block_size"])
DAMPENING_FRAC          = float(PROFILE_CFG[PROFILE]["dampening_frac"])
ACTORDER                = str(PROFILE_CFG[PROFILE]["actorder"])
WEIGHT_SYMMETRIC        = bool(PROFILE_CFG[PROFILE]["symmetric"])
CONCATENATE_DATA        = bool(PROFILE_CFG[PROFILE]["concatenate_data"])
PAD_TO_MAX_LENGTH       = bool(PROFILE_CFG[PROFILE]["pad_to_max_length"])
CALIB_LEN_MIX           = PROFILE_CFG[PROFILE]["len_mix"]
CALIB_TEXT_MODE         = PROFILE_CFG[PROFILE]["text_mode"]
KOREAN_FOCUS            = bool(PROFILE_CFG[PROFILE]["korean_focus"])
KOREAN_MIN_RATIO        = float(PROFILE_CFG[PROFILE]["korean_min_ratio"])

TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]  # 정확도 우선이면 유지가 유리한 경우 많음

STREAM_SHUFFLE_BUFFER = 100000
CALIB_MAX_CHARS  = 40000
CALIB_MIN_TOKENS = 64


# ---------------------------
# 3) 유틸 함수
# ---------------------------
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.set_grad_enabled(False)
torch.backends.cuda.matmul.allow_tf32 = True

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def safe_load_tokenizer(path, **kwargs):
    """fix_mistral_regex는 절대 켜지 않음"""
    try:
        return AutoTokenizer.from_pretrained(path, fix_mistral_regex=False, **kwargs)
    except TypeError:
        return AutoTokenizer.from_pretrained(path, **kwargs)

def hangul_ratio(s: str) -> float:
    if not s:
        return 0.0
    h = len(re.findall(r"[가-힣]", s))
    return h / max(1, len(s))

def build_length_stratified_calib_dataset_streaming(
    tokenizer,
    dataset_id,
    split,
    seed,
    num_samples,
    max_seq_len,
    shuffle_buffer=100000,
    len_mix=(0.25, 0.55, 0.20),
    text_mode="PROMPT",
    max_chars=40000,
    min_tokens=64,
    korean_focus=True,
    korean_min_ratio=0.08,
):
    """
    - streaming shuffle로 계속 샘플링
    - 토큰 길이 기준 short/mid/long 버킷을 목표 비율로 채움
    - ✅ korean_focus는 "강제 필터"가 아니라 "우선 선발 + 부족분 fallback" 방식(실패 방지)
    """
    assert abs(sum(len_mix) - 1.0) < 1e-6, "len_mix must sum to 1.0"

    ns = int(num_samples * len_mix[0])
    nm = int(num_samples * len_mix[1])
    nl = num_samples - ns - nm

    # 길이 임계값
    t_short = min(int(max_seq_len * 0.25), 1024)
    t_mid   = min(int(max_seq_len * 0.50), 2048)

    ds_stream = load_dataset(dataset_id, split=split, streaming=True)
    ds_stream = ds_stream.shuffle(seed=seed, buffer_size=shuffle_buffer)

    buckets = {"short": [], "mid": [], "long": []}
    need = {"short": ns, "mid": nm, "long": nl}

    # 우선/대체 풀
    extra_pool = []          # (korean 조건 충족하는데 bucket이 이미 찼을 때)
    fallback_pool = []       # (korean 조건 미달 텍스트) -> 나중에 부족분 채우기용

    extra_cap = max(4096, int(num_samples * 0.75))
    fallback_cap = max(4096, int(num_samples * 1.25))

    picked = 0
    scanned = 0
    max_scan = max(250000, num_samples * 250)

    for ex in ds_stream:
        scanned += 1
        if scanned > max_scan and sum(need.values()) > 0:
            print(f"[CALIB][WARN] max_scan reached ({max_scan}). backfill from pools.")
            break

        conv = ex.get("conversations", None)
        if not conv:
            continue

        if text_mode == "PROMPT":
            add_gen = True
        elif text_mode == "FULL":
            add_gen = False
        else:
            add_gen = (picked % 2 == 0)  # MIX

        text = tokenizer.apply_chat_template(conv, add_generation_prompt=add_gen, tokenize=False)
        if (not text) or (len(text) > max_chars):
            continue

        # korean focus: 우선 선발. 부족하면 fallback_pool에서 채움
        if korean_focus and hangul_ratio(text) < korean_min_ratio:
            if len(fallback_pool) < fallback_cap:
                fallback_pool.append(text)
            continue

        ids = tokenizer(
            text,
            add_special_tokens=False,
            truncation=True,
            max_length=max_seq_len,
            return_attention_mask=False,
        )["input_ids"]
        L = len(ids)
        if L < min_tokens:
            continue

        if L <= t_short:
            key = "short"
        elif L <= t_mid:
            key = "mid"
        else:
            key = "long"

        if need[key] > 0:
            buckets[key].append(text)
            need[key] -= 1
            picked += 1
        else:
            if len(extra_pool) < extra_cap:
                extra_pool.append(text)

        if sum(need.values()) == 0:
            break

    texts = buckets["short"] + buckets["mid"] + buckets["long"]

    # 1차: extra_pool로 부족분 채움(여기까지는 korean 조건 충족)
    if len(texts) < num_samples:
        need_n = num_samples - len(texts)
        texts += extra_pool[:need_n]

    # 2차: 그래도 부족하면 fallback_pool(한국어 조건 미달)로 채움
    if len(texts) < num_samples:
        need_n = num_samples - len(texts)
        texts += fallback_pool[:need_n]

    if len(texts) < num_samples:
        raise RuntimeError(
            f"[CALIB] samples 부족: got {len(texts)}/{num_samples} "
            f"(need_left={need}, extra_pool={len(extra_pool)}, fallback_pool={len(fallback_pool)}, scanned={scanned})"
        )

    rng = random.Random(seed)
    rng.shuffle(texts)
    ds = Dataset.from_list([{"text": t} for t in texts[:num_samples]])

    print(
        f"[CALIB] size={len(ds)} | mode={text_mode} | max_len={max_seq_len} | "
        f"mix={len_mix} | buckets=({len(buckets['short'])},{len(buckets['mid'])},{len(buckets['long'])}) | "
        f"fallback_used={max(0, len(ds)-len(buckets['short'])-len(buckets['mid'])-len(buckets['long'])-min(len(extra_pool), max(0, num_samples-(len(buckets['short'])+len(buckets['mid'])+len(buckets['long'])))))} | "
        f"scanned={scanned}"
    )
    return ds

def patch_json(path, patch_fn):
    if not os.path.exists(path):
        return
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    before = json.dumps(data, sort_keys=True)
    patch_fn(data)
    after = json.dumps(data, sort_keys=True)
    if before != after:
        tmp = path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        os.replace(tmp, path)

def patch_runtime_config(out_dir, tok):
    cfg_path = os.path.join(out_dir, "config.json")
    gen_path = os.path.join(out_dir, "generation_config.json")
    tok_cfg  = os.path.join(out_dir, "tokenizer_config.json")

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    def _patch_cfg(cfg):
        cfg["use_cache"] = True
        cfg["eos_token_id"] = tok.eos_token_id
        cfg["pad_token_id"] = tok.pad_token_id
        # ✅ torch_dtype 명시(로더가 auto일 때 흔들리는 케이스 방지)
        if cfg.get("torch_dtype") is None:
            cfg["torch_dtype"] = "float16"
        if tok.bos_token_id is not None:
            cfg["bos_token_id"] = tok.bos_token_id

    def _patch_gen(gen):
        gen["use_cache"] = True
        gen["eos_token_id"] = tok.eos_token_id
        gen["pad_token_id"] = tok.pad_token_id
        if tok.bos_token_id is not None:
            gen["bos_token_id"] = tok.bos_token_id

    def _patch_tok(tc):
        tc["fix_mistral_regex"] = False
        # ✅ (중요) 런타임에서도 tail 보존되게 저장물에 left 강제
        tc["truncation_side"] = "left"

    patch_json(cfg_path, _patch_cfg)
    patch_json(gen_path, _patch_gen)
    patch_json(tok_cfg, _patch_tok)

def patch_tokenizer_pre_tokenizer_sequence(out_dir):
    tok_json = os.path.join(out_dir, "tokenizer.json")
    if not os.path.exists(tok_json):
        return
    with open(tok_json, "r", encoding="utf-8") as f:
        data = json.load(f)
    pt = data.get("pre_tokenizer", None)
    if isinstance(pt, dict) and pt.get("type") == "ByteLevel":
        data["pre_tokenizer"] = {"type": "Sequence", "pretokenizers": [pt]}
        with open(tok_json, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)
        print("[TOKENIZER PATCH] wrapped ByteLevel -> Sequence")

def copy_code_assets_from_base(base_dir, out_dir):
    if not os.path.isdir(base_dir):
        return
    for fn in os.listdir(base_dir):
        if fn.endswith((".py", ".jinja")):
            src = os.path.join(base_dir, fn)
            dst = os.path.join(out_dir, fn)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)

def ensure_chat_template(out_dir, src_model_dir):
    src = os.path.join(src_model_dir, "chat_template.jinja")
    dst = os.path.join(out_dir, "chat_template.jinja")
    if (not os.path.exists(dst)) and os.path.exists(src):
        shutil.copy2(src, dst)
        print("[INFO] copied chat_template.jinja")

def build_submit_zip_from_model_dir(model_dir, zip_name="submit"):
    submit_root = "/content/submit"
    submit_model = os.path.join(submit_root, "model")

    shutil.rmtree(submit_root, ignore_errors=True)
    os.makedirs(submit_model, exist_ok=True)

    for name in os.listdir(model_dir):
        src = os.path.join(model_dir, name)
        dst = os.path.join(submit_model, name)
        if os.path.isdir(src):
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)

    zip_path = f"{zip_name}.zip"
    if os.path.exists(zip_path):
        os.remove(zip_path)

    shutil.make_archive(
        base_name=zip_name,
        format="zip",
        root_dir=submit_root,
        base_dir="."
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        top_levels = set([n.split("/")[0] for n in z.namelist() if n])
        if top_levels != {"model"}:
            raise RuntimeError(f"ZIP 최상위가 model만이어야 합니다. 현재: {top_levels}")

    return zip_path

def offline_load_test(model_dir):
    env_keys = ["HF_HOME", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"]
    old = {k: os.environ.get(k) for k in env_keys}

    tmp_cache = "/content/hf_offline_cache_tmp"
    shutil.rmtree(tmp_cache, ignore_errors=True)
    os.makedirs(tmp_cache, exist_ok=True)

    try:
        os.environ["HF_HOME"] = tmp_cache
        os.environ["TRANSFORMERS_OFFLINE"] = "1"
        os.environ["HF_DATASETS_OFFLINE"] = "1"

        tok = safe_load_tokenizer(model_dir, trust_remote_code=True, local_files_only=True, use_fast=True)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        # ✅ 저장물이 left인지 확인
        print("[OFFLINE TEST] tokenizer.truncation_side =", getattr(tok, "truncation_side", None))

        model = AutoModelForCausalLM.from_pretrained(
            model_dir,
            trust_remote_code=True,
            local_files_only=True,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        ).cuda().eval()

        msgs = [{"role":"user","content":"한 문장으로 자기소개 해줘."}]
        prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        inputs = tok(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")

        with torch.inference_mode():
            out = model.generate(
                **inputs,
                do_sample=False,
                num_beams=1,
                max_new_tokens=64,
                use_cache=True,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id,
            )

        gen = out[0, inputs["input_ids"].shape[-1]:]
        print("[OFFLINE OUTPUT]", tok.decode(gen, skip_special_tokens=True).strip()[:200])

        del model
        torch.cuda.empty_cache(); gc.collect()
        print("[OFFLINE TEST] ✅ PASS")
    finally:
        for k, v in old.items():
            if v is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = v

def _walk(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield (k, v)
            yield from _walk(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from _walk(v)

def verify_quantization_saved(out_dir, expected_bits=4, expected_group_size=None):
    print("[VERIFY] checking quantization artifacts...")

    cfg_path = os.path.join(out_dir, "config.json")
    if not os.path.exists(cfg_path):
        raise RuntimeError("[VERIFY] config.json not found (save_pretrained 결과 확인 필요)")

    with open(cfg_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    qc = cfg.get("quantization_config") or cfg.get("compression_config")
    if qc is None:
        raise RuntimeError(
            "[VERIFY] config.json에 quantization_config가 없습니다.\n"
            " - model.save_pretrained(save_compressed=True)로 저장됐는지 확인하세요.\n"
            " - 또는 oneshot/modify_save_pretrained 적용이 안 됐을 수 있습니다."
        )

    found_bits = False
    found_gs = (expected_group_size is None)

    for k, v in _walk(qc):
        if k == "num_bits" and v == expected_bits:
            found_bits = True
        if expected_group_size is not None and k == "group_size" and v == expected_group_size:
            found_gs = True

    if not found_bits:
        raise RuntimeError(f"[VERIFY] config.json.quantization_config에서 num_bits={expected_bits}를 못 찾음")
    if not found_gs:
        raise RuntimeError(f"[VERIFY] config.json.quantization_config에서 group_size={expected_group_size}를 못 찾음")

    st_files = [f for f in os.listdir(out_dir) if f.endswith(".safetensors")]
    if not st_files:
        raise RuntimeError("[VERIFY] safetensors 파일이 없음 (저장 실패 가능)")

    p0 = os.path.join(out_dir, st_files[0])
    size_mb = os.path.getsize(p0) / (1024 * 1024)
    print(f"[VERIFY] ✅ config.json contains expected num_bits/group_size")
    print(f"[VERIFY] safetensors: {st_files[0]} ({size_mb:.1f} MB)")

    try:
        with safe_open(p0, framework="pt", device="cpu") as sf:
            keys = list(sf.keys())
            meta = sf.metadata() or {}

        patterns = ["weight_scale", "weight_zero_point", "qweight", "qzeros", "scales", "g_idx"]
        found = [ptn for ptn in patterns if any(ptn in k for k in keys)]
        if found:
            print(f"[VERIFY] quant tensor hint(s) in keys: {found}")
        else:
            print("[VERIFY][INFO] safetensors key 힌트는 못 찾았지만, config.json 기준 검증은 통과했습니다.")

        if meta:
            print("[VERIFY] safetensors metadata keys:", list(meta.keys())[:10])

    except Exception as e:
        print("[VERIFY][WARN] safetensors key scan skipped:", repr(e))

    # ✅ tokenizer_config에 truncation_side=left가 들어갔는지 확인
    tok_cfg = os.path.join(out_dir, "tokenizer_config.json")
    if os.path.exists(tok_cfg):
        tc = json.load(open(tok_cfg, "r", encoding="utf-8"))
        if tc.get("truncation_side") != "left":
            raise RuntimeError("[VERIFY] tokenizer_config.json truncation_side가 left가 아닙니다 (점수 급락 위험)")
        print("[VERIFY] ✅ tokenizer_config.json truncation_side=left")

    print("[VERIFY] ✅ PASS")

def call_oneshot_no_silent_drop(**kwargs):
    sig = inspect.signature(oneshot)
    supported = set(sig.parameters.keys())

    # oneshot이 **kwargs를 받는 형태면 엄격검증이 무의미할 수 있어 예외 처리
    has_var_kw = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values())
    if not has_var_kw:
        unknown = [k for k in kwargs.keys() if k not in supported]
        if unknown:
            raise TypeError(
                "oneshot()이 아래 인자를 지원하지 않습니다(조용히 드랍 안 함): "
                f"{unknown}\n"
                f"지원 인자 목록: {sorted(list(supported))}"
            )
    return oneshot(**kwargs)


# ---------------------------
# 4) Main
# ---------------------------
t_all0 = time.time()
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    raise RuntimeError("이 스크립트는 GPU(cuda) 환경을 전제로 합니다.")

print(f"[INFO] llmcompressor={llmcompressor.__version__}")
print(f"[INFO] device={device} | PROFILE={PROFILE}")
print(f"[INFO] calib={NUM_CALIBRATION_SAMPLES} | max_len={MAX_SEQUENCE_LENGTH} | calib_bs={CALIB_BATCH_SIZE}")
print(f"[INFO] WEIGHT group_size={GPTQ_GROUP_SIZE} | GPTQ block_size={GPTQ_BLOCK_SIZE} | damp={DAMPENING_FRAC} | actorder={ACTORDER}")
print(f"[INFO] concatenate_data={CONCATENATE_DATA} | pad_to_max_length={PAD_TO_MAX_LENGTH}")
print(f"[INFO] CALIB_TEXT_MODE={CALIB_TEXT_MODE} | KOREAN_FOCUS={KOREAN_FOCUS} (min_ratio={KOREAN_MIN_RATIO})")
print(f"[INFO] MODEL_ID={MODEL_ID}")

if not os.path.isdir(MODEL_ID):
    raise FileNotFoundError(f"MODEL_ID 폴더가 존재하지 않습니다: {MODEL_ID}")

print("[INFO] tokenizer load...")
tokenizer = safe_load_tokenizer(MODEL_ID, trust_remote_code=True, local_files_only=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ✅ (중요) 런타임 점수 보호: tokenizer 자체를 left로 세팅(저장까지 반영)
tokenizer.truncation_side = "left"

print("[INFO] model load...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    local_files_only=True,
    low_cpu_mem_usage=True,
).to(device).eval()

# max_position_embeddings clamp
if hasattr(model.config, "max_position_embeddings") and model.config.max_position_embeddings is not None:
    mpe = int(model.config.max_position_embeddings)
    if MAX_SEQUENCE_LENGTH > mpe:
        print(f"[WARN] MAX_SEQUENCE_LENGTH({MAX_SEQUENCE_LENGTH}) > max_position_embeddings({mpe}). Clamping.")
        MAX_SEQUENCE_LENGTH = mpe

# GPTQ 캘리브레이션 중 캐시 끄기(안정)
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False
if hasattr(model, "generation_config") and hasattr(model.generation_config, "use_cache"):
    model.generation_config.use_cache = False

print("[INFO] build calibration dataset...")
ds = build_length_stratified_calib_dataset_streaming(
    tokenizer=tokenizer,
    dataset_id=DATASET_ID,
    split=DATASET_SPLIT,
    seed=SEED,
    num_samples=NUM_CALIBRATION_SAMPLES,
    max_seq_len=MAX_SEQUENCE_LENGTH,
    shuffle_buffer=STREAM_SHUFFLE_BUFFER,
    len_mix=CALIB_LEN_MIX,
    text_mode=CALIB_TEXT_MODE,
    max_chars=CALIB_MAX_CHARS,
    min_tokens=CALIB_MIN_TOKENS,
    korean_focus=KOREAN_FOCUS,
    korean_min_ratio=KOREAN_MIN_RATIO,
)
print("[INFO] calib dataset size:", len(ds))

# ---------------------------
# 4-1) GPTQ recipe (config_groups로 weights.group_size 명시)
# ---------------------------
CONFIG_GROUPS = {
    "group_0": {
        "targets": TARGETS,
        "input_activations": None,
        "output_activations": None,
        "weights": {
            "num_bits": 4,
            "type": "int",
            "symmetric": WEIGHT_SYMMETRIC,
            "strategy": "group",
            "group_size": GPTQ_GROUP_SIZE,
        },
    }
}

recipe = [
    GPTQModifier(
        config_groups=CONFIG_GROUPS,
        ignore=IGNORE,
        block_size=GPTQ_BLOCK_SIZE,
        dampening_frac=DAMPENING_FRAC,
        actorder=ACTORDER,
        offload_hessians=False,
    )
]

print("[INFO] GPTQ start...")
torch.cuda.empty_cache(); gc.collect()
t0 = time.time()

out = call_oneshot_no_silent_drop(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=len(ds),
    tokenizer=tokenizer,
    shuffle_calibration_samples=False,
    batch_size=CALIB_BATCH_SIZE,
    concatenate_data=CONCATENATE_DATA,   # ✅ v4.1: 기본 False
    pad_to_max_length=PAD_TO_MAX_LENGTH,
)

if out is not None:
    model = out

t1 = time.time()
print(f"[INFO] GPTQ done: {t1-t0:.1f}s")

# ---------------------------
# 4-2) Save
# ---------------------------
print("[INFO] save compressed model...")
shutil.rmtree(OUT_DIR_LOCAL, ignore_errors=True)
os.makedirs(OUT_DIR_LOCAL, exist_ok=True)

model.save_pretrained(
    OUT_DIR_LOCAL,
    save_compressed=True,
    safe_serialization=True,
)
tokenizer.save_pretrained(OUT_DIR_LOCAL)

copy_code_assets_from_base(MODEL_ID, OUT_DIR_LOCAL)
ensure_chat_template(OUT_DIR_LOCAL, MODEL_ID)

patch_runtime_config(OUT_DIR_LOCAL, tokenizer)
patch_tokenizer_pre_tokenizer_sequence(OUT_DIR_LOCAL)

print("[INFO] final model dir:", OUT_DIR_LOCAL)
print("[INFO] model dir files (head):", sorted(os.listdir(OUT_DIR_LOCAL))[:30], "...")

# ✅ 저장 산출물 검증
verify_quantization_saved(
    OUT_DIR_LOCAL,
    expected_bits=4,
    expected_group_size=GPTQ_GROUP_SIZE,
)

# ✅ 오프라인 자기완결 로드 테스트
print("[INFO] offline self-contained test...")
offline_load_test(OUT_DIR_LOCAL)

# ---------------------------
# 4-3) ZIP + Drive copy
# ---------------------------
print("[INFO] build submit.zip...")
zip_path = build_submit_zip_from_model_dir(OUT_DIR_LOCAL, zip_name=ZIP_NAME)
size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"[INFO] zip created: {zip_path} ({size_mb:.1f} MB)")

try:
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
    shutil.rmtree(DRIVE_MODEL_DIR, ignore_errors=True)
    shutil.copytree(OUT_DIR_LOCAL, DRIVE_MODEL_DIR)
    shutil.copy2(zip_path, DRIVE_ZIP_PATH)
    print("[INFO] copied model to:", DRIVE_MODEL_DIR)
    print("[INFO] copied zip to  :", DRIVE_ZIP_PATH)
except Exception as e:
    print("[WARN] Drive copy skipped/failed:", repr(e))

cfg = json.load(open(os.path.join(OUT_DIR_LOCAL, "config.json"), "r", encoding="utf-8"))
print("[CONFIG CHECK]", {k: cfg.get(k) for k in ["torch_dtype","max_position_embeddings","attn_implementation","use_cache","pad_token_id","eos_token_id"]})

t_all1 = time.time()
print(f"[INFO] TOTAL elapsed: {(t_all1-t_all0)/60:.2f} min")
print("[DONE] zip=", zip_path)

[INFO] llmcompressor=0.9.0.1
[INFO] device=cuda | PROFILE=H100_SCORE
[INFO] calib=32768 | max_len=4096 | calib_bs=1
[INFO] WEIGHT group_size=32 | GPTQ block_size=128 | damp=0.01 | actorder=static
[INFO] concatenate_data=False | pad_to_max_length=False
[INFO] CALIB_TEXT_MODE=MIX | KOREAN_FOCUS=False (min_ratio=0.05)
[INFO] MODEL_ID=/content/drive/MyDrive/data/lg_8th/base_model
[INFO] tokenizer load...


`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] model load...
[INFO] build calibration dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[CALIB] size=32768 | mode=MIX | max_len=4096 | mix=(0.25, 0.55, 0.2) | buckets=(8192,18022,3608) | fallback_used=0 | scanned=1000000
[INFO] calib dataset size: 32768
[INFO] GPTQ start...


...

2026-02-16T16:43:32.189361+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-16T16:43:32.453102+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
[INFO] GPTQ done: 5121.4s
[INFO] save compressed model...
2026-02-16T16:43:32.454975+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:01, 146.85it/s]


[TOKENIZER PATCH] wrapped ByteLevel -> Sequence
[INFO] final model dir: /content/model_quant
[INFO] model dir files (head): ['chat_template.jinja', 'config.json', 'generation_config.json', 'merges.txt', 'model.safetensors', 'recipe.yaml', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json'] ...
[VERIFY] checking quantization artifacts...
[VERIFY] ✅ config.json contains expected num_bits/group_size
[VERIFY] safetensors: model.safetensors (1374.1 MB)
[VERIFY] quant tensor hint(s) in keys: ['weight_scale']
[VERIFY] safetensors metadata keys: ['format']
[VERIFY] ✅ tokenizer_config.json truncation_side=left
[VERIFY] ✅ PASS
[INFO] offline self-contained test...
[OFFLINE TEST] tokenizer.truncation_side = left


Compressing model: 210it [00:00, 1706.76it/s]


[OFFLINE OUTPUT] "안녕하세요! 다양한 경험과 지식을 바탕으로 창의적인 문제 해결자로서 성장해왔습니다."
[OFFLINE TEST] ✅ PASS
[INFO] build submit.zip...
[INFO] zip created: submit.zip (1133.0 MB)
[INFO] copied model to: /content/drive/MyDrive/data/lg_8th/try0216_submit_v41/model
[INFO] copied zip to  : /content/drive/MyDrive/data/lg_8th/try0216_submit_v41/submit.zip
[CONFIG CHECK] {'torch_dtype': 'float16', 'max_position_embeddings': 65536, 'attn_implementation': None, 'use_cache': True, 'pad_token_id': 0, 'eos_token_id': 361}
[INFO] TOTAL elapsed: 118.83 min
[DONE] zip= submit.zip
